# Build the tissue layer — the body's cell-cell + cross-organ wiring

For each organ: the cell types, **who signals whom** (ligand→receptor), which channels are druggable,
the **downstream response inside receiver cells**, and the cross-tissue **endocrine (hormone) axes**.
Independent of the cell layer — **CPU is fine, ~a few minutes**. No GPU needed.


## 1 · Setup — clone repo


In [ ]:
import os, sys, urllib.request
BR = 'claude/vectorize-gex-propensity-NRqBW'
if not os.path.exists('colab/build_tissue_model.py'):
    os.system(f'git clone -q --branch {BR} https://github.com/nikku03/cell.git')
    if os.path.isdir('cell') and os.path.exists('cell/colab/build_tissue_model.py'): os.chdir('cell')
assert os.path.exists('colab/build_tissue_model.py'), 'repo not cloned correctly'
H='data/external_data/human'; OUT='outputs/orphan'
os.makedirs(H, exist_ok=True); os.makedirs(OUT, exist_ok=True)


## 2 · Restore the Drive data that powers the richer layers
HPA + Omnipath are downloaded fresh below; these Drive files add druggable channels + downstream response.


In [ ]:
import shutil
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('Drive mount skipped:', e)
DV='/content/drive/MyDrive/virtual_cell_data'; CM='/content/drive/MyDrive/cell_model'
def _get(src, dst):
    if os.path.isfile(src) and not (os.path.exists(dst) and os.path.getsize(dst)>1000):
        shutil.copy2(src, dst); print('restored', os.path.basename(dst))
for fn in ['dgidb.tsv','signor.tsv','collectri.tsv','cellphonedb.csv','cpdb_gene.csv','cpdb_protein.csv']:
    _get(f'{DV}/human_raw/{fn}', f'{H}/{fn}')
# NicheNet ligand->target (a computed output from the full build) enables downstream-inside-receiver wiring
for src in [f'{CM}/nichenet_ligand_targets.json', f'{DV}/cell_build/nichenet_ligand_targets.json']:
    _get(src, f'{OUT}/nichenet_ligand_targets.json')


## 3 · Download the two core sources (HPA single-cell + Omnipath ligand-receptor)


In [ ]:
op=urllib.request.build_opener(); op.addheaders=[('User-Agent','Mozilla/5.0')]; urllib.request.install_opener(op)
SRC={'hpa_sc.tsv.zip':'https://www.proteinatlas.org/download/tsv/rna_single_cell_type.tsv.zip',
     'omnipath_ligrec.tsv':'https://omnipathdb.org/interactions?datasets=ligrecextra&types=post_translational&genesymbols=yes&organisms=9606&fields=sources,references&format=tsv'}
for fn,url in SRC.items():
    p=f'{H}/{fn}'
    if os.path.exists(p) and os.path.getsize(p)>10000: print('have',fn); continue
    print('downloading',fn,'...'); urllib.request.urlretrieve(url,p); print('  ',os.path.getsize(p)//1024,'KB')
# optional: CellChat pathway families (for colouring channels by pathway) — skip if it fails
try:
    cc='https://omnipathdb.org/annotations?resources=CellChatDB&genesymbols=yes&format=tsv'
    if not os.path.exists(f'{H}/omnipath_cellchat.tsv'): urllib.request.urlretrieve(cc, f'{H}/omnipath_cellchat.tsv')
except Exception as e: print('cellchat optional -> skip:', e)


## 4 · Build the tissue model + explorer


In [ ]:
import subprocess
subprocess.run([sys.executable,'colab/build_tissue_model.py'], check=True)
subprocess.run([sys.executable,'colab/build_tissue_explorer.py'], check=True)


## 5 · Verify + save to Drive + download


In [ ]:
import json
T=json.load(open(f'{OUT}/tissue_model.json'))
tis=T.get('tissues',{})
nch=sum(len(v.get('links',v)) if isinstance(v,dict) else 0 for v in tis.values()) if isinstance(tis,dict) else 0
print('tissues:', list(tis.keys()) if isinstance(tis,dict) else tis)
print('ligand-receptor pairs:', T.get('n_lr'), '| downstream receptors:', len(T.get('downstream',{})),
      '| druggable LR genes:', len(T.get('drugs',{})), '| endocrine axes:', len(T.get('endocrine',[])),
      '| nichenet ligands:', len(T.get('nichenet',{})))
assert isinstance(tis,dict) and len(tis)>0, 'no tissues built — check HPA/Omnipath downloads'
CM='/content/drive/MyDrive/cell_model'; os.makedirs(CM, exist_ok=True)
for f in ['tissue_model.json','tissue_explorer.html']:
    if os.path.exists(f'{OUT}/{f}'): shutil.copy(f'{OUT}/{f}', f'{CM}/{f}'); print('saved ->', f'{CM}/{f}')
try:
    from google.colab import files; files.download(f'{OUT}/tissue_model.json')
except Exception as e: print('download skipped:', e)
print('Send me tissue_model.json for the cell<->tissue coupling.')


## What you built
Pick a tissue · click a cell type (who it signals) · click an arrow (ligand→receptor channels) ·
knock out a gene (e.g. COL1A1, CTLA4) to see which cell-cell signals break — in `tissue_explorer.html`.
